<a href="https://colab.research.google.com/github/eduardokern/ML/blob/face_detection/notebooks/face_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Criando um sistema de reconhecimento facial do zero</h1>

O objetivo principal deste projeto é trabalhar com as bibliotecas e frameworks estudados e analisados em nossas aulas. Neste sentido, a proposta padrão envolve um sistema de detecção e reconhecimento de faces, utilizando o framework TensorFlow em conjuntos com as bibliotecas que o projetista julgue necessárias, de forma ilimitada.  

Por meio da Figura 1 é possível visualizar o resultado esperado para o modelo proposto, devendo detectar e reconhecer mais de uma face ao mesmo tempo.  
Para isso você deve:

1. Utilizar uma rede de detecção treinada para detectar faces.
2. Utilizar uma rede de classificação para classificar a face detectada.

![Figura 1: Detecção e reconhecimento facial.](https://drive.google.com/uc?export=view&id=1nFCZd-FR0jIYAvDbDbVpDt6ansjEkLIA)

Para realizar este projeto, você pode utilizar os seguintes trabalhos de referência:
Detecção Facial:
https://colab.research.google.com/drive/1QnC7lV7oVFk5OZCm75fqbLAfD9qBy9bw?usp=sharing

Detecção e classificação de objetos:  
https://colab.research.google.com/drive/1xdjyBiY75MAVRSjgmiqI7pbRLn58VrbE?usp=sharing

<h3>Mount Google Drive to get datasets</h3>

In [1]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

datasets_path = '/content/drive/MyDrive/Colab/ML/datasets'

Mounted at /content/drive


<h3>Required imports</h3>

In [15]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout
from keras.models import Sequential
from keras.preprocessing import image

<h3>Helper functions</h3>

In [3]:
# helper function to load image and return it and input vector
def get_image(path):
  try:
    img = image.load_img(path, target_size=(224, 224), keep_aspect_ratio=True)
    data = image.img_to_array(img)
    data = np.expand_dims(data, axis=0)
    data = preprocess_input(data)
    return img, data
  except Exception as e:
    print(f'Error loading: {path}, {e}')
    return None, None

In [22]:
# helper function to load detection dataset, image data and bbox
def load_dataset(path):
    dataset = []
    with open(f'{path}/_annotations.csv') as file:
        file.readline()
        for line in file.readlines():
            values = line.replace('\n', '').split(',')
            if len(values) > 1:
                filename = values[0]
                width = int(values[1])
                height = int(values[2])
                label = values[3]
                xmin = float(values[4])
                ymin = float(values[5])
                xmax = float(values[6])
                ymax = float(values[7])

                image_path = f'{path}/{filename}'
                image, data = get_image(image_path)
                dataset.append({'x':np.array(data[0]), 'y':[xmin/width, ymin/height, xmax/width, ymax/height], 'image': image})

    return dataset

<h3>FaceDetection class</h3>

It contais a CNN to detect all faces and their bbox and a VGG16 neural network to classify the faces detected.

Both NN can be trained with different datasets.

In [25]:
class FaceDetection:
  def __init__(self, detection_train_data, detection_val_data):
    self._detection_model(detection_train_data[0].shape[1:])
    self._train_detection(detection_train_data, detection_val_data, 10, 128)

  def _detection_model(self, input_shape):
    print(input_shape)
    self._detection_model = Sequential()

    self._detection_model.add(Input(shape=input_shape))
    self._detection_model.add(Conv2D(32, (3, 3), activation='relu'))
    self._detection_model.add(Conv2D(64, (3, 3), activation='relu'))
    self._detection_model.add(Flatten())
    self._detection_model.add(Dense(128, activation='relu'))
    # Output layer for bounding box coordinates
    self._detection_model.add(Dense(4, activation='linear'))

    self._detection_model.summary()


  def _train_detection(self, train_data, val_data, epochs, batch_size):
    print('compile')
    self._detection_model.compile(optimizer='adam', loss='mse', metrics=['accuracy'])
    print('fit')
    self._detection_model.fit(
        train_data[0],
        train_data[1],
        validation_data=val_data,
        epochs=epochs,
        batch_size=batch_size)


  def detect(self, image):
    return self._detection_model.predict(np.array([image]))


  def train_classification(self, num_classes, train_data, val_data, epochs, batch_size):
    vgg = keras.applications.VGG16(weights='imagenet', include_top=True)
    inp = vgg.input

    # make a new softmax layer with num_classes neurons
    new_classification_layer = Dense(num_classes, activation='softmax')

    # connect our new layer to the second to last layer in VGG, and make a reference to it
    out = new_classification_layer(vgg.layers[-2].output)

    # create a new network between inp and out
    self._classification_model = Model(inp, out)

    # make all layers untrainable by freezing weights (except for last layer)
    for l, layer in enumerate(self._classification_model.layers[:-1]):
        layer.trainable = False

    # ensure the last layer is trainable/not frozen
    for l, layer in enumerate(self._classification_model.layers[-1:]):
        layer.trainable = True

    self._classification_model.compile(
        loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    self._classification_model.fit(
        train_data[0],
        train_data[1],
        validation_data=val_data,
        epochs=epochs,
        batch_size=batch_size)


  def predict(self, image):
    faces = self._detection_model.predict(np.array([image]))


Load face detection dataset to train the face detection CNN to identify all faces and their bbox in an image.

In [23]:
detection_dataset_path = f'{datasets_path}/FaceDetection'
detection_train = load_dataset(f'{detection_dataset_path}/train')
detection_valid = load_dataset(f'{detection_dataset_path}/valid')
detection_test = load_dataset(f'{detection_dataset_path}/test')

x_detection_train, y_detection_train = np.array([t["x"] for t in detection_train]), [t["y"] for t in detection_train]
x_detection_train = x_detection_train.astype('float32') /255.
y_detection_train = np.array(y_detection_train, dtype="float32")

x_detection_valid, y_detection_valid = np.array([t["x"] for t in detection_valid]), [t["y"] for t in detection_valid]
x_detection_valid = x_detection_valid.astype('float32') /255.
y_detection_valid = np.array(y_detection_valid, dtype="float32")


In [ ]:
detector = FaceDetection(
    (x_detection_train, y_detection_train),
    (x_detection_valid, y_detection_valid))

image, data = get_image(f'{datasets_path}/big3/big3/big3-1.jfif')
result = detector.detect(data)
print(result)

(224, 224, 3)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_14 (Conv2D)                   │ (None, 222, 222, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_15 (Conv2D)                   │ (None, 220, 220, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_5 (Flatten)                  │ (None, 3097600)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 128)                 │     396,492,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 4)                   │             516 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 396,512,836 (1.48 GB)

 Trainable params: 396,512,836 (1.48 GB)

 Non-trainable params: 0 (0.00 B)

compile
fit
Epoch 1/10
